In [1]:
import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
from torchvision import models
from torchvision.transforms import v2
from torchvision.io import decode_image, ImageReadMode

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [2]:
trainDf = pd.read_csv('./data/train_labels.csv')
testDf = pd.read_csv('./data/test_labels.csv')
valDf = pd.read_csv('./data/val_labels.csv')

In [3]:
K = set(trainDf.breed) | set(testDf.breed) | set(valDf.breed)
print(len(K),K)

120 {'chesapeake_bay_retriever', 'malamute', 'groenendael', 'brabancon_griffon', 'pekinese', 'redbone', 'brittany_spaniel', 'whippet', 'kerry_blue_terrier', 'gordon_setter', 'rhodesian_ridgeback', 'miniature_schnauzer', 'basenji', 'miniature_poodle', 'cardigan', 'otterhound', 'toy_terrier', 'american_staffordshire_terrier', 'lakeland_terrier', 'english_springer', 'sussex_spaniel', 'miniature_pinscher', 'airedale', 'lhasa', 'italian_greyhound', 'chow', 'maltese_dog', 'boxer', 'greater_swiss_mountain_dog', 'giant_schnauzer', 'japanese_spaniel', 'old_english_sheepdog', 'black-and-tan_coonhound', 'scotch_terrier', 'pug', 'bull_mastiff', 'tibetan_mastiff', 'briard', 'african_hunting_dog', 'border_collie', 'dandie_dinmont', 'curly-coated_retriever', 'scottish_deerhound', 'french_bulldog', 'pomeranian', 'shetland_sheepdog', 'yorkshire_terrier', 'leonberg', 'keeshond', 'soft-coated_wheaten_terrier', 'affenpinscher', 'bloodhound', 'norwich_terrier', 'ibizan_hound', 'eskimo_dog', 'borzoi', 'seal

In [ ]:
efficientNet = models.efficientnet_v2_s(weights=None)
efficientNet.to(device)

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 24, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(24, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): FusedMBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(24, 24, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
            (1): BatchNorm2d(24, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
        )
        (stochastic_depth): StochasticDepth(p=0.0, mode=row)
      )
      (1): FusedMBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(24, 24, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
            (1): BatchNorm2d(24, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  

: 

In [15]:
resize = v2.Resize((100, 100), antialias=True)
# decode_image gives RGB, channel-first (C,H,W) uint8 tensors
imgs = torch.stack([
    resize(decode_image(f'./data/train/images/{p}.jpg', mode=ImageReadMode.RGB))
    for p in trainDf.id
]).to(device)  # uint8, N,C,H,W
print(imgs.shape, imgs.dtype)

torch.Size([7155, 3, 100, 100]) torch.uint8


In [16]:
x = imgs.float() / 255
mean = x.mean(dim=(0, 2, 3))
std = x.std(dim=(0, 2, 3))
del x
print("Mean of the training set is:", mean, "Standard Deviation of training set is:", std)

Mean of the training set is: tensor([0.4772, 0.4528, 0.3928], device='cuda:0') Standard Deviation of training set is: tensor([0.2534, 0.2480, 0.2523], device='cuda:0')


In [ ]:
norm = v2.Normalize(mean.tolist(), std.tolist())
train_tf = v2.Compose([v2.RandomHorizontalFlip(p=0.5), v2.RandomCrop(100, padding=12)])

def prep(xb, train=False):
    if train:
        xb = train_tf(xb)
    return norm(xb.float() / 255)

xb = prep(imgs[:64], train=True)
print(xb.shape, xb.mean(dim=(0, 2, 3)), xb.std(dim=(0, 2, 3)))